<!-- NB_DOC_INTRO_V1 -->
# AI Engineering — Local LLMs (Ollama, llama.cpp, vLLM)

> 📚 **Doc thematique** : [docs/09_AI_ENGINEERING.md](docs/09_AI_ENGINEERING.md) (AI Engineering)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Faire tourner des LLMs en local** plutot que via API (OpenAI / Anthropic) : avantages (privacy, cout marginal nul, offline, fine-tuning, no censure), tradeoffs (latence, qualite top-tier moins accessible, setup). Ce notebook compare les outils 2025 (Ollama, llama.cpp, vLLM, LM Studio, text-generation-inference, GPT4All) et explique le **choix de modele** (parametres, quantization, GPU / Mac M-series / CPU).

**Demo** : si Ollama est installe, on appelle un modele local via l'API HTTP compatible OpenAI. Sinon, code de reference pret a executer.

Versions : `ollama (CLI) >= 0.3`, `llama.cpp` recent (2024-2025).

## Plan

1. Pourquoi local plutot que SaaS
2. Trade-offs local vs API
3. Comparatif outils (Ollama, llama.cpp, vLLM, LM Studio, ...)
4. Quantizations (FP16, INT8, GGUF Q4/Q2, AWQ, GPTQ)
5. **Ollama** : install + usage CLI + API HTTP (executable si installe)
6. **llama.cpp** : compile-based, ultime
7. **vLLM** : prod GPU haut throughput
8. **Choix du modele 2024-2025** (Llama 3.2, Mistral, Qwen, Phi, DeepSeek, Gemma)
9. Patterns d'integration (OpenAI client → Ollama)
10. Pieges et anti-patterns
11. References


## 1. Pourquoi local plutot que SaaS ?

| Critere | Local | API (OpenAI/Anthropic) |
|---|---|---|
| **Qualite top** | Llama-3.1-70B / Mistral Large ~ GPT-3.5/GPT-4o-mini | GPT-4, Claude Sonnet 4 = etat de l'art |
| **Vitesse** | Lent sur gros modeles sans GPU lourd | Tres rapide |
| **Cout marginal** | 0 (apres matos) | $/M tokens |
| **Setup** | Complexe | API key, c'est tout |
| **Privacy** | ✅ totale | Donnees envoyees |
| **Customisation** | Totale (FT possible) | Limitee |
| **Pas de censure provider** | Sauf modeles "aligned" | Politique du provider |
| **Offline** | ✅ | ❌ |


## 2. Comparatif outils — Local LLMs

| Outil | Backend | UX | Cas ideal |
|---|---|---|---|
| **Ollama** | llama.cpp | Excellent (`ollama run llama3`) | Local dev, prototype, API OpenAI-like |
| **llama.cpp** | C++ | CLI / serveur HTTP | Production CPU/Mac M-series, controle total |
| **vLLM** | PyTorch + Triton kernels | API OpenAI-compatible | Production GPU haut debit, PagedAttention |
| **LM Studio** | llama.cpp | GUI desktop | Non-tech, demo |
| **text-generation-inference** (HF) | Rust + PyTorch | Serveur HF | Production GPU, integration HF |
| **GPT4All** | llama.cpp | GUI + Python | Apps Python desktop |
| **Jan / Msty / Enchanted** | llama.cpp | GUI moderne | Alternative LM Studio |
| **MLX-LM** (Apple) | Apple MLX | Python | Mac M-series natif |


## 3. Quantizations — comprendre les acronymes

Quantization = **reduire la precision des poids** pour gain RAM / vitesse, avec perte de qualite controlee.

| Format | RAM (sur 7B) | Tokens/sec (RTX 4090) | Qualite |
|---|---:|---:|---|
| **FP32** | 28 GB | n/a | Reference (rarement utilise) |
| **FP16 / BF16** | 14 GB | ~ 60 | ~ identique a FP32 |
| **INT8** | 7 GB | ~ 70 | ~ 99% qualite |
| **GGUF Q8_0** | 7.5 GB | ~ 75 | ~ identique FP16 |
| **GGUF Q5_K_M** | 5 GB | ~ 80 | ~ 99% qualite |
| **GGUF Q4_K_M** | 4.5 GB | ~ 85 | ~ 98% qualite (★ recommande) |
| **GGUF Q3_K_M** | 3.5 GB | ~ 90 | ~ 95% qualite |
| **GGUF Q2_K** | 2.5 GB | ~ 95 | ~ 85% qualite (degrade) |
| **AWQ** (4-bit) | 4 GB | ~ 100 (GPU) | ~ 99% (GPU only) |
| **GPTQ** (4-bit) | 4 GB | ~ 95 (GPU) | ~ 99% (GPU only) |

> 🎯 **Best practice** : **Q4_K_M** est le sweet spot quasi universel (CPU + GPU). Q5_K_M si tu as la RAM. Q8 si tu veux zero perte.

### Sur Mac M-series (Apple Silicon)
- llama.cpp avec **Metal** backend = surprenamment competitif (Llama 7B Q4 = ~ 30 tok/s sur M2)
- **MLX-LM** (Apple) = encore plus optimise mais ecosystem plus jeune


## 4. Ollama — la voie simple

Ollama = wrapper user-friendly autour de llama.cpp + un registry de modeles + API HTTP compatible OpenAI.

### Installation
```bash
# Linux / Mac
curl -fsSL https://ollama.com/install.sh | sh

# Windows : installer depuis https://ollama.com/
```

### Usage CLI
```bash
ollama pull llama3.2:3b              # telecharge (~2 GB en GGUF Q4)
ollama run llama3.2:3b               # mode chat interactif
ollama list                          # liste les modeles locaux
ollama rm llama3.2:3b                # supprime

# API HTTP (compatible OpenAI sur :11434)
curl http://localhost:11434/api/generate -d '{"model": "llama3.2:3b", "prompt": "Hello"}'
```


In [ ]:
# Demo Python : tenter d'appeler Ollama local si dispo
import requests

OLLAMA_URL = "http://localhost:11434"

def ollama_available() -> bool:
    try:
        r = requests.get(f"{OLLAMA_URL}/api/tags", timeout=2)
        return r.status_code == 200
    except Exception:
        return False

def list_local_models() -> list[str]:
    r = requests.get(f"{OLLAMA_URL}/api/tags")
    return [m["name"] for m in r.json().get("models", [])]

if ollama_available():
    print(f"Ollama up. Modeles locaux : {list_local_models()}")
else:
    print("Ollama indisponible (pas demarre / pas installe).")
    print("Installer : https://ollama.com/  puis 'ollama serve'")

In [ ]:
# Demo : si un modele est dispo, on l'appelle
import requests

def ollama_chat(model: str, prompt: str, system: str = None) -> str:
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    r = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={"model": model, "messages": messages, "stream": False},
        timeout=60,
    )
    return r.json()["message"]["content"]

if ollama_available():
    models = list_local_models()
    if models:
        small_model = models[0]
        answer = ollama_chat(
            model=small_model,
            prompt="Explique le RAG en 2 phrases.",
            system="Tu es un assistant technique concis.",
        )
        print(f"Modele : {small_model}")
        print(f"Reponse :\n{answer}")
    else:
        print("Aucun modele Ollama installe. Tente : ollama pull llama3.2:3b")
else:
    print("Ollama non dispo. Pseudo-output :")
    print('  ollama_chat("llama3.2:3b", "Explique le RAG en 2 phrases.")')
    print('  → "Le RAG (Retrieval-Augmented Generation) combine recherche..."')

### Utiliser le client OpenAI vers Ollama (drop-in replacement)

C'est l'astuce qui rend Ollama immediatement utilisable depuis le code existant : son API est **compatible OpenAI**.

```python
from openai import OpenAI

# Pointer le client OpenAI vers Ollama au lieu d'api.openai.com
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

r = client.chat.completions.create(
    model="llama3.2:3b",
    messages=[{"role": "user", "content": "Hello"}],
)
print(r.choices[0].message.content)
```

Permet de **basculer entre local et SaaS** en changeant juste `base_url` + `api_key`.


### Custom model avec Modelfile (equivalent Dockerfile)
```
# Modelfile
FROM llama3.2:3b
PARAMETER temperature 0.3
PARAMETER num_ctx 4096
SYSTEM "Tu es un assistant Python expert qui repond en francais avec des exemples de code."
```
```bash
ollama create my-py-assistant -f Modelfile
ollama run my-py-assistant
```


## 5. llama.cpp — hardcore mais ultime

Compilation depuis source ou install via Homebrew. Donne le controle total + meilleure perf si tu sais tuner.

```bash
brew install llama.cpp                       # Mac
# ou : git clone https://github.com/ggerganov/llama.cpp && cd llama.cpp && make

# Telecharger un GGUF depuis HuggingFace
huggingface-cli download bartowski/Mistral-7B-Instruct-v0.3-GGUF \
  Mistral-7B-Instruct-v0.3-Q4_K_M.gguf --local-dir ~/models

# CLI
llama-cli -m ~/models/Mistral-7B-Instruct-v0.3-Q4_K_M.gguf -p "Explique RAG" -n 200

# Serveur OpenAI-compatible
llama-server -m ~/models/Mistral-7B-Instruct-v0.3-Q4_K_M.gguf \
  --host 0.0.0.0 --port 8080 \
  --n-gpu-layers 35           # offload N layers sur GPU (Mac M-series, NVIDIA)
```

### Bindings Python
```python
# pip install -q llama-cpp-python
from llama_cpp import Llama
llm = Llama(model_path="model.gguf", n_ctx=4096, n_gpu_layers=35)
r = llm("Explique le RAG", max_tokens=200)
print(r["choices"][0]["text"])
```


## 6. vLLM — production GPU haut throughput

vLLM = serveur d'inference avec **PagedAttention** (Kwon et al. 2023) → 24× plus de throughput que HF transformers naif.

```bash
pip install -q vllm    # ~ Linux + CUDA recommande

# Mode programmatique
from vllm import LLM, SamplingParams
llm = LLM(model="meta-llama/Meta-Llama-3-8B-Instruct", dtype="bfloat16",
          gpu_memory_utilization=0.9, max_model_len=8192)
outputs = llm.generate(["Hello", "Bonjour"], SamplingParams(temperature=0.7, max_tokens=200))
for o in outputs:
    print(o.outputs[0].text)

# Mode serveur (OpenAI-compatible)
python -m vllm.entrypoints.openai.api_server \
  --model meta-llama/Meta-Llama-3-8B-Instruct \
  --gpu-memory-utilization 0.9 \
  --port 8000
# Avec quantization : --quantization awq / --quantization gptq
```


## 7. Choix du modele 2024-2025

### Petits (jusqu'a 3B) — CPU friendly
| Modele | Taille | Cas |
|---|---:|---|
| **Llama-3.2-1B / 3B** (Meta) | 1B / 3B | Tres leger, mobile |
| **Phi-3.5-mini** (Microsoft) | 3.8B | Excellent rapport qualite/taille |
| **Gemma-2-2B** (Google) | 2.6B | Solide, multilingue |
| **Qwen2.5-3B** (Alibaba) | 3B | Excellent en code & multilingue |

### Moyens (7-9B) — GPU consumer (RTX 4060+) ou Mac M2+
| Modele | Cas |
|---|---|
| **Mistral-7B-Instruct-v0.3** | Reference, bon FR |
| **Llama-3.1-8B-Instruct** | Tres bon EN, tool use |
| **Qwen2.5-7B** | Excellent code, multilingue |
| **Mistral-Nemo-12B** | Compromis taille/qualite (12B) |

### Grands (30B-70B) — GPU pro ou Mac M-series 64+ GB
| Modele | Cas |
|---|---|
| **Llama-3.1-70B-Instruct** | Top open-source, niveau GPT-4o-mini |
| **Qwen2.5-72B** | Concurrent direct Llama-3 |
| **Mistral-Large-2** | Quasi GPT-4, Apache 2.0 |
| **DeepSeek-V3** | 671B MoE, top niveau (require beaucoup de VRAM) |

### Specialises
| Cas | Modele |
|---|---|
| **Code** | DeepSeek-Coder-V2, Qwen2.5-Coder, Codestral |
| **Math** | DeepSeek-Math, Llemma |
| **Multimodal (vision)** | LLaVA-Next, Qwen2-VL, Pixtral |
| **Embedding** | bge-large-en-v1.5, e5-large-v2, jina-embeddings-v3 |
| **Re-ranking** | bge-reranker-v2-m3 |

### Sur CPU pur (Ryzen 5 / i5 + 16 GB RAM)
Q4 d'un 7B fonctionne : ~ 5-10 tok/s, **acceptable** pour POC ou usage personnel.

### Sur Mac M2/M3 (Metal)
Etonnamment competitif via Ollama / llama.cpp. Llama-3 8B Q4 : ~ 30 tok/s sur M2 Pro.


## 8. Patterns d'integration

| Use case | Stack recommandee |
|---|---|
| **Dev local rapide** | Ollama + Llama-3.2-3B ou Phi-3.5-mini |
| **Agent RAG en entreprise** | vLLM + Qwen2.5-14B + Qdrant |
| **App desktop user-friendly** | LM Studio + GGUF Q4 |
| **API maison** | llama-server (llama.cpp) ou vLLM |
| **Coding assistant** | Continue.dev + DeepSeek-Coder via Ollama |
| **Mobile / offline** | llama.cpp avec Phi-3.5 ou Llama-3.2-1B |
| **Embeddings local** | sentence-transformers (bge-m3) — pas un LLM mais souvent associe |

### Pattern hybride : dev local, prod SaaS
```python
import os
from openai import OpenAI

if os.getenv("USE_LOCAL_LLM"):
    client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    MODEL = "llama3.2:3b"
else:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    MODEL = "gpt-4o-mini"

# Memes appels apres ca
```


## 9. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correction |
|---|---|
| Charger un 70B en FP16 sur 24 GB VRAM (impossible) | Quantize Q4 ou utiliser modele plus petit |
| Comparer Q2 quantize avec FP16 d'un autre | Q2 = qualite degradee, comparer a quantization equivalente |
| Mesurer la perf en startup latency | Charger le modele = lent, generation = OK ; mesurer steady-state |
| Oublier `num_ctx` — defaut souvent 2048 alors que le modele supporte 32k | Specifier explicitement |
| Tester avec temperature elevee et conclure "le modele hallucine" | Baisser temperature a 0.1 pour eval factuelle |
| Lancer `ollama pull modele:latest` sans verifier l'espace disque | Verifier `du -sh ~/.ollama/models` |
| Ne pas streamer | Mauvaise UX (utilisateur attend longtemps) |
| Pas de prompt template / chat template | Modele instruct fonctionne mal sans le bon template (chatml, llama3, mistral) |
| Pas de seed / temperature pour reproductibilite | `temperature=0` + `seed=42` |
| Tester un modele instruct comme un completion model | Modeles `*-Instruct` veulent un format chat (system + user + assistant) |

## 10. References
- **Ollama** : https://ollama.com/  •  Library : https://ollama.com/library
- **llama.cpp** : https://github.com/ggerganov/llama.cpp
- **vLLM** : https://docs.vllm.ai/
- **LM Studio** : https://lmstudio.ai/
- **Open LLM Leaderboard** : https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard
- **MTEB** (embeddings leaderboard) : https://huggingface.co/spaces/mteb/leaderboard
- **HuggingFace GGUF** : https://huggingface.co/models?library=gguf
- **TheBloke / bartowski** (auteurs GGUF de reference) : https://huggingface.co/bartowski

Voir aussi :
- [AI_Prompt_Engineering.ipynb](./AI_Prompt_Engineering.ipynb)
- [AI_LLM_Finetuning_PEFT_LoRA.ipynb](./AI_LLM_Finetuning_PEFT_LoRA.ipynb)
- [NLP_LangChain_RAG.ipynb](./NLP_LangChain_RAG.ipynb)


<!-- ENRICH_2025_V1 -->

## 🔥 LLMs locaux — etat de l'art fin 2024 / 2025

### Top modeles open-weights (decembre 2024)

| Modele | Taille | Forces | Cas |
|---|---:|---|---|
| **Llama 3.3 70B** (Meta) | 70B | Top open-weights, multilingue | Best general purpose |
| **Qwen 2.5 72B** (Alibaba) | 72B | Excellent code + math | Code, multilingue (FR ok) |
| **DeepSeek-V3** | 671B MoE | SOTA open, **gratuit** API | Quand on a la VRAM |
| **DeepSeek-R1** | 671B MoE | **Raisonnement explicite** (CoT) | Math, code, logique |
| **Mistral Large 2** | 123B | Europeen, Apache 2.0 | Production EU |
| **Llama 3.2 1B / 3B** | 1B / 3B | Mobile, edge | Smartphones |
| **Phi-3.5-mini 3.8B** | 3.8B | Excellent rapport qualite/taille | CPU friendly |
| **Gemma 2 2B / 9B / 27B** | 2-27B | Google, multilingue | Generaliste leger |

### Modeles **specialises** code

| Modele | Note |
|---|---|
| **Qwen 2.5 Coder 32B** | Top open code, niveau Sonnet 3.5 |
| **DeepSeek Coder V2** | 236B MoE, excellent |
| **Codestral 22B** (Mistral) | Coding assistant |
| **StarCoder 2** | Code completion |

### Modeles **multimodaux** (vision + texte)

| Modele | Note |
|---|---|
| **Llama 3.2 Vision 11B / 90B** | Open vision-language |
| **Qwen 2 VL 72B** | Top open multimodal |
| **Pixtral 12B** (Mistral) | Vision Apache 2.0 |
| **LLaVA OneVision** | Open recherche |

## 🔥 Outils 2024-2025 — au-dela d'Ollama

### Pour le **serveur** local
- **vLLM** : standard production GPU (PagedAttention)
- **LMDeploy** (Shanghai AI Lab) : alternative vLLM, tres rapide
- **TGI** (HuggingFace) : Rust + PyTorch, integre HF Hub
- **llama-cpp-python** : CPU + Metal (Mac), GGUF
- **MLX-LM** (Apple) : Mac M-series natif, very fast

### Pour le **client / chat UI**
- **Open WebUI** : ChatGPT-like interface, multi-model
- **LM Studio** : desktop GUI macOS/Linux/Windows
- **Jan** : alternative LM Studio, open source
- **Msty** : recent, jolie UX
- **Continue.dev** : VS Code extension, code assistant

### Pour **automatiser**
- **Ollama Python lib** : `pip install ollama`
- **LiteLLM** : abstraction multi-provider (Ollama, OpenAI, Anthropic, ...)
- **Open Interpreter** : LLM execute du code local (sandbox)

## 🔥 Quantizations 2025

GGUF Q4_K_M reste le sweet spot, mais nouveaux formats :

| Format | Note |
|---|---|
| **GGUF Q4_K_M** | Standard llama.cpp, RAM-efficient |
| **AWQ** | GPU, qualite +1-2% vs GGUF Q4 |
| **GPTQ** | GPU, similar AWQ |
| **EXL2** | exllamav2, le plus rapide GPU |
| **FP8** (Ada Lovelace / H100) | Half FP16, qualite quasi identique |
| **NF4** (QLoRA) | Pour FT, pas inference |

## 💡 Quel modele pour quel use case (decembre 2024) ?

| Cas | Modele |
|---|---|
| **Dev local Mac M2** | Llama 3.2 3B ou Phi-3.5 via Ollama |
| **Coding assistant** | Qwen 2.5 Coder 14B via Continue.dev |
| **API local (1 utilisateur)** | Llama 3.1 8B / Mistral 7B via Ollama |
| **API equipe (5-10 users)** | Qwen 2.5 14B via vLLM + Open WebUI |
| **Production GPU haut debit** | Llama 3.3 70B via vLLM + LB |
| **Mobile / edge** | Llama 3.2 1B + llama.cpp |
| **Privacy stricte (no cloud)** | Mistral Large 2 self-hosted |
| **Multilingue FR** | Mistral Nemo 12B / Qwen 2.5 |
| **Math / raisonnement** | DeepSeek R1 (si VRAM) ou Qwen 2.5 Math |
| **Vision** | Llama 3.2 Vision ou Pixtral |
